# 01 · Preparación de datos y flujo Roboflow
**IAM-811** · Laboratorio Final de Hito — Sistema Inteligente de Control de Peaje

Este notebook arma los tres datasets del proyecto y los deja listos para revisar en **Roboflow**:

| Dataset | Tipo de proyecto Roboflow | Clases / keypoints | Lo usa |
|---|---|---|---|
| `vehiculos` | *Instance Segmentation* | auto, bus, camion, minibus, moto | `02_M2_segmentacion_vehiculos.ipynb` |
| `llantas` | *Keypoint Detection* | llanta · 5 kp: centro, contacto, superior, izquierda, derecha | `03_M3_keypoints_llantas_ejes.ipynb` |
| `placas` | *Object Detection* | placa | `04_M1_placas_ocr.ipynb` |

**Flujo recomendado**
1. Graba videos cortos de vehículos (vista lateral o 3/4 para ejes; frontal/trasera para placas) o junta fotos.
2. Este notebook extrae frames y **auto-etiqueta** con YOLOE (zero-shot, como en los notebooks de clase).
3. Sube el zip a Roboflow y **corrige** las etiquetas (un modelo entrenado solo con etiquetas automáticas copia sus errores).
4. Genera una versión (augmentations: flip horizontal, brillo ±25 %, blur leve) y expórtala en formato **YOLOv11**.
5. Los notebooks 02–04 descargan esa versión con la API de Roboflow.

> La API key de Roboflow **no** va en el código: guárdala en *Colab → Secretos (icono de llave)* como `ROBOFLOW_API_KEY`.

**Antes de empezar:** *Entorno de ejecución → Cambiar tipo de entorno → GPU T4*.

## 1. Instalación

In [ ]:
!nvidia-smi -L || echo "Sin GPU"
!pip -q install -U ultralytics==8.4.218 roboflow==1.7.3
!pip -q install 'git+https://github.com/ultralytics/CLIP.git'

## 2. Configuración

In [ ]:
import os, glob, random, shutil, json, cv2, numpy as np, torch
from ultralytics import YOLOE
DEVICE = 0 if torch.cuda.is_available() else 'cpu'

BASE = '/content/datos'
DIR_VIDEOS, DIR_FRAMES = f'{BASE}/videos', f'{BASE}/frames'
for d in (DIR_VIDEOS, DIR_FRAMES):
    os.makedirs(d, exist_ok=True)

# Orden alfabetico = el orden con que Roboflow exporta las clases (el servidor las lee por NOMBRE)
CLASES_VEH = ['auto', 'bus', 'camion', 'minibus', 'moto']
PROMPTS_VEH = {'motorcycle': 'moto', 'motorbike': 'moto', 'car': 'auto', 'sedan': 'auto', 'suv': 'auto',
               'minibus': 'minibus', 'van': 'minibus', 'truck': 'camion', 'cargo truck': 'camion', 'bus': 'bus'}
PROMPTS_LLANTA = ['wheel', 'tire']
PROMPTS_PLACA = ['license plate']
KPT_NOMBRES = ['centro', 'contacto', 'superior', 'izquierda', 'derecha']
FLIP_IDX = [0, 1, 2, 4, 3]
PORC_VAL = 0.2

## 3. Subir videos / fotos
Sube archivos `.mp4` o un `.zip` de fotos. Alternativa: monta Google Drive y copia desde ahí.

In [ ]:
from google.colab import files
for nombre in files.upload().keys():
    if nombre.lower().endswith('.zip'):
        !unzip -qo "$nombre" -d {DIR_FRAMES}
    elif nombre.lower().endswith(('.mp4', '.mov', '.avi', '.mkv')):
        shutil.move(nombre, DIR_VIDEOS)
    else:
        shutil.move(nombre, DIR_FRAMES)

## 4. Extraer frames de los videos (2 fps, igual que la app móvil)

In [ ]:
FPS_EXTRACCION = 2
for v in glob.glob(f'{DIR_VIDEOS}/*'):
    cap = cv2.VideoCapture(v)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    paso, i, n = max(1, int(round(fps / FPS_EXTRACCION))), 0, 0
    while True:
        ok, fr = cap.read()
        if not ok:
            break
        if i % paso == 0:
            cv2.imwrite(f'{DIR_FRAMES}/{os.path.splitext(os.path.basename(v))[0]}_{i:06d}.jpg', fr)
            n += 1
        i += 1
    print(os.path.basename(v), '->', n, 'frames')
EXT = ('.jpg', '.jpeg', '.png', '.webp')
IMGS = sorted(p for p in glob.glob(DIR_FRAMES + '/**/*', recursive=True) if p.lower().endswith(EXT))
print('Total de imágenes:', len(IMGS))

## 5. Modelo YOLOE (vocabulario abierto) para auto-etiquetar

In [ ]:
nombres = list(PROMPTS_VEH) + PROMPTS_LLANTA + PROMPTS_PLACA
yoloe = YOLOE('yoloe-11l-seg.pt')
yoloe.set_classes(nombres, yoloe.get_text_pe(nombres))

def detectar(img):
    r = yoloe.predict(img, conf=0.15, imgsz=1024, device=DEVICE, verbose=False)[0]
    out = []
    if r.boxes is None or len(r.boxes) == 0:
        return out
    polis = r.masks.xy if r.masks is not None else [None] * len(r.boxes)
    for c, s, b, p in zip(r.boxes.cls.cpu().numpy().astype(int), r.boxes.conf.cpu().numpy(),
                          r.boxes.xyxy.cpu().numpy(), polis):
        out.append({'nombre': r.names[c], 'conf': float(s), 'box': b.tolist(), 'poly': p})
    return out

def kps_de_mascara(mask):
    # 5 keypoints geométricos de la llanta (mismo método del notebook keypoints_llantas del curso)
    cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if not cnts:
        return None
    c = max(cnts, key=cv2.contourArea)
    if cv2.contourArea(c) < 8:
        return None
    pts = c.reshape(-1, 2).astype(float)
    cx, cy = cv2.fitEllipse(c)[0] if len(pts) >= 5 else pts.mean(0)
    return np.array([[cx, cy], pts[pts[:, 1].argmax()], pts[pts[:, 1].argmin()],
                     pts[pts[:, 0].argmin()], pts[pts[:, 0].argmax()]])

## 6. Auto-etiquetado en formato YOLO (segmentación, pose y detección)

In [ ]:
DS = {k: f'{BASE}/dataset_{k}' for k in ('vehiculos', 'llantas', 'placas')}
for d in DS.values():
    shutil.rmtree(d, ignore_errors=True)
    for s in ('train', 'val'):
        os.makedirs(f'{d}/images/{s}'); os.makedirs(f'{d}/labels/{s}')

random.seed(0)
orden = IMGS[:]
random.shuffle(orden)
n_val = max(1, int(len(orden) * PORC_VAL))
conteo = {'vehiculos': 0, 'llantas': 0, 'placas': 0}
for i, ruta in enumerate(orden):
    split = 'val' if i < n_val else 'train'
    img = cv2.imread(ruta)
    if img is None:
        continue
    H, W = img.shape[:2]
    lv, ll, lp = [], [], []
    for d in detectar(img):
        n = d['nombre']
        if n in PROMPTS_VEH and d['conf'] >= 0.30 and d['poly'] is not None and len(d['poly']) >= 3:
            cls = CLASES_VEH.index(PROMPTS_VEH[n])
            lv.append(f'{cls} ' + ' '.join(f'{x / W:.6f} {y / H:.6f}' for x, y in d['poly']))
        elif n in PROMPTS_LLANTA and d['poly'] is not None and len(d['poly']) >= 3:
            m = np.zeros((H, W), np.uint8)
            cv2.fillPoly(m, [d['poly'].astype(np.int32)], 1)
            k = kps_de_mascara(m)
            if k is None:
                continue
            x1, y1, x2, y2 = d['box']
            x1, y1 = min(x1, k[:, 0].min()), min(y1, k[:, 1].min())
            x2, y2 = max(x2, k[:, 0].max()), max(y2, k[:, 1].max())
            caja = [(x1 + x2) / 2 / W, (y1 + y2) / 2 / H, (x2 - x1) / W, (y2 - y1) / H]
            kp = ' '.join(f'{min(max(x / W, 0), 1):.6f} {min(max(y / H, 0), 1):.6f} 2' for x, y in k)
            ll.append('0 ' + ' '.join(f'{v:.6f}' for v in caja) + ' ' + kp)
        elif n in PROMPTS_PLACA and d['conf'] >= 0.25:
            x1, y1, x2, y2 = d['box']
            lp.append(f'0 {(x1 + x2) / 2 / W:.6f} {(y1 + y2) / 2 / H:.6f} {(x2 - x1) / W:.6f} {(y2 - y1) / H:.6f}')
    for clave, lineas in (('vehiculos', lv), ('llantas', ll), ('placas', lp)):
        cv2.imwrite(f'{DS[clave]}/images/{split}/{i:05d}.jpg', img)
        open(f'{DS[clave]}/labels/{split}/{i:05d}.txt', 'w').write('\n'.join(lineas))
        conteo[clave] += len(lineas)
print('Etiquetas generadas:', conteo)

def escribir_yaml(ruta, nombres_cls, extra=''):
    with open(f'{ruta}/data.yaml', 'w') as f:
        f.write(f'path: {ruta}\ntrain: images/train\nval: images/val\n{extra}names:\n')
        for i, n in enumerate(nombres_cls):
            f.write(f'  {i}: {n}\n')

escribir_yaml(DS['vehiculos'], CLASES_VEH)
escribir_yaml(DS['llantas'], ['llanta'], f'kpt_shape: [5, 3]\nflip_idx: {FLIP_IDX}\n')
escribir_yaml(DS['placas'], ['placa'])
for k in DS:
    !cd {BASE} && zip -qr dataset_{k}.zip dataset_{k}
!ls -lh {BASE}/*.zip

## 7. Vista previa de las etiquetas automáticas de vehículos

In [ ]:
import matplotlib.pyplot as plt
todas = glob.glob(f"{DS['vehiculos']}/images/train/*.jpg")
muestras = random.sample(todas, k=min(4, len(todas)))
fig, ax = plt.subplots(1, max(1, len(muestras)), figsize=(6 * max(1, len(muestras)), 5), squeeze=False)
for a, p in zip(ax[0], muestras):
    img = cv2.imread(p)
    H, W = img.shape[:2]
    for l in open(p.replace('/images/', '/labels/').replace('.jpg', '.txt')).read().split('\n'):
        if not l:
            continue
        v = l.split()
        pts = (np.array(v[1:], float).reshape(-1, 2) * [W, H]).astype(np.int32)
        cv2.polylines(img, [pts], True, (0, 255, 0), 2)
        cv2.putText(img, CLASES_VEH[int(v[0])], tuple(int(t) for t in pts[0]), 0, 1, (0, 255, 0), 2)
    a.imshow(img[:, :, ::-1]); a.axis('off')
plt.show()

## 8. Subir a Roboflow para corregir
> **Llantas (keypoints):** al crear el proyecto *Keypoint Detection* define la clase `llanta` con los puntos
> **en este orden exacto**: `centro`, `contacto`, `superior`, `izquierda`, `derecha`. Roboflow exporta los keypoints en el
> orden en que se definieron; si cambia el orden, el conteo de ejes del servidor sale mal. Si Roboflow no acepta importar
> las etiquetas de pose del zip, puedes entrenar el notebook 03 directamente con `FUENTE = 'zip'` (ya vienen en ese orden)
> y corregir en Roboflow las imágenes con errores.

**Opción manual (la más simple):** descarga los zip y arrástralos a un proyecto nuevo en
<https://app.roboflow.com> (elige el tipo de proyecto de la tabla del inicio). Roboflow reconoce el formato YOLO.
Corrige, genera una versión y expórtala como **YOLOv11**.

**Opción por API:** usa el SDK con tu clave guardada en los Secretos de Colab.

In [ ]:
# files.download(f'{BASE}/dataset_vehiculos.zip')
# files.download(f'{BASE}/dataset_llantas.zip')
# files.download(f'{BASE}/dataset_placas.zip')
SUBIR_POR_API = False
if SUBIR_POR_API:
    from google.colab import userdata
    import roboflow
    rf = roboflow.Roboflow(api_key=userdata.get('ROBOFLOW_API_KEY'))
    ws = rf.workspace()
    for clave, tipo in (('vehiculos', 'instance-segmentation'), ('placas', 'object-detection')):
        ws.upload_dataset(DS[clave], f'peaje-{clave}', project_type=tipo, num_workers=8)
    # El proyecto de keypoints (llantas) se sube por la interfaz web arrastrando el zip.

## 9. Guardar en Google Drive (opcional)

In [ ]:
GUARDAR_DRIVE = False
if GUARDAR_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    !mkdir -p /content/drive/MyDrive/peaje/datasets && cp {BASE}/*.zip /content/drive/MyDrive/peaje/datasets/